# Modelling clothing shop reviews

In this notebook first we will split the data, then create pipelines for num, cat and text data. After that we will merge this pipelines and use ML models for predictions. At the end we will also fine tune our model.

## Libraries import and data preparation 

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import spacy
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from transformers import AutoTokenizer, AutoModel
import torch
from sklearn.preprocessing import FunctionTransformer

nlp = spacy.load('en_core_web_sm')

In [ ]:
data = pd.read_csv("reviews.csv")
num_cols = ["Age", "Positive Feedback Count", "Recommended IND"]
cat_cols = ["Division Name", "Department Name", "Class Name"]
text_cols = ["Title", "Review Text"]
data = data.drop(["Clothing ID"], axis=1)

X = data.drop('Recommended IND', axis=1)
y = data['Recommended IND'].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, shuffle=True, random_state=42)

## Preparing Pipelines


### Numerical Features Pipeline

In [ ]:
num_pipeline = Pipeline([
    (
        'scaler',
        StandardScaler(),
    ),
])

### Categorical Features Pipeline

In [ ]:
cat_pipeline = Pipeline([
    (
        'ordinal_encoder',
        OrdinalEncoder(
            handle_unknown='use_encoded_value',
            unknown_value=-1,
        )
    ),    
    (
        'cat_encoder',
        OneHotEncoder(
            sparse_output=False,
            handle_unknown='ignore',
        )
    ),
])


### Text Feature Pipeline 

In the text pipeline we will create two custom features. First one will count good and bad words already chosen like good, perfect or bad.

In [ ]:
class WordCount(BaseEstimator, TransformerMixin):
    def __init__(self):
        return

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return [len(text.split(" ")) for text in X]



In [ ]:
# Sentiment Analysis transformer

class Sentiment(BaseEstimator, TransformerMixin):
    def __init__(self, model_name='bert-base-uncased'):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)

    def fit(self,X,y=None):
        return self
    
    def transform(self, X):
        inputs = self.tokenizer(X, return_tensors='pt', padding= True)
        with torch.no_grad():
            outputs = self.model(**inputs, output_hidden_states = True)
        embeddings = outputs.hidden_states[-1][:,0,:].numpy()
        return embeddings